# FUNGI+ Pipeline
Clear shards before restarting: `rm -rf data/output/phase3_expansive_search/shards/`

## Setup

In [1]:
import gc
import json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# Local machine: src/ is a subdirectory of this notebook's own directory
# (was a hardcoded HPC /scratch path). Assumes the notebook is run with its
# working directory at the FUNGI/ project root.
SRC_DIR = str(Path.cwd() / "src")
import sys
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

CONFIG_PATH = "fungi_config.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

FUNGI_MODE = cfg.get("fungi_mode", "organic").lower()
if str(FUNGI_MODE).lower() == "organic": FUNGI_MODE = "biologic"  # exp_011 legacy alias
assert FUNGI_MODE in ("organic", "synthetic"), FUNGI_MODE
print(f"Run mode: {FUNGI_MODE.upper()}")

Run mode: ORGANIC


In [2]:
# Paths and output directories
RAW_GRAPH_PATH = Path(cfg["input"]["graph_path"])
SC_DATA_PATH   = Path(cfg["input"]["sc_data_path"])
OUTPUT_ROOT    = Path(cfg["output"]["root_dir"])

for phase in cfg["output"]["phases"]:
    (OUTPUT_ROOT / phase).mkdir(parents=True, exist_ok=True)

print(f"Run        : fungi_run")
print(f"Config     : {CONFIG_PATH}")
print(f"Mode       : {FUNGI_MODE.upper()}")
print(f"Graph      : {RAW_GRAPH_PATH.name}")
print(f"Expression : {SC_DATA_PATH.name}")
print(f"Output     : {OUTPUT_ROOT}")

Run        : fungi_run
Config     : fungi_config.yaml
Mode       : ORGANIC
Graph      : RPE1_5k_essential_selftrain_dense_graph.parquet
Expression : RPE1_5k_essential_train_metacell.h5ad
Output     : ..\DATA\FUNGI_outputs


## Phase 1: Ingestion

In [3]:
import scanpy as sc
import scipy.sparse as sp

# Load expression first so the gene universe and ordering are fixed by adata
adata = sc.read_h5ad(str(SC_DATA_PATH))
N_GENES = adata.n_vars
gene_to_idx = {g: i for i, g in enumerate(adata.var_names)}
print(f"Expression : {adata.n_obs:,} cells x {adata.n_vars:,} genes")


Expression : 17,695 cells x 5,000 genes


In [4]:
# Load the dense parent graph (parquet) and map it onto the adata gene order.
# Column names follow the PSGRN output convention.
gdf = pd.read_parquet(str(RAW_GRAPH_PATH))
_cols = {c.lower(): c for c in gdf.columns}
src_col = _cols.get("regulator", _cols.get("source", gdf.columns[0]))
tgt_col = _cols.get("target", gdf.columns[1])
w_col   = _cols.get("importance", _cols.get("weight", gdf.columns[2]))
print(f"Columns: source='{src_col}' target='{tgt_col}' weight='{w_col}'")

rows = gdf[src_col].map(gene_to_idx).to_numpy()
cols = gdf[tgt_col].map(gene_to_idx).to_numpy()
vals = gdf[w_col].to_numpy(dtype=np.float64)
mask = ~(pd.isna(rows) | pd.isna(cols))
raw_sparse_mat = sp.csr_matrix(
    (vals[mask], (rows[mask].astype(int), cols[mask].astype(int))),
    shape=(N_GENES, N_GENES))
raw_sparse_mat.eliminate_zeros()

# Standard GRN: no experimental confidence columns. If your pipeline attaches
# md_score / stability columns, load experimental_df here instead.
experimental_df = None
print(f"Parent GRN : {N_GENES:,} genes | {raw_sparse_mat.nnz:,} edges | "
      f"density {raw_sparse_mat.nnz / N_GENES**2:.4%}")
if (~mask).any():
    print(f"  dropped {int((~mask).sum()):,} edges with genes not in adata.var_names")


Columns: source='Regulator' target='Target' weight='Importance'
Parent GRN : 5,000 genes | 24,995,000 edges | density 99.9800%


## Gene features and kernel switches

In [5]:
from utils import build_gene_features, build_kernel_flags

gene_features = build_gene_features(adata, n_components=50, seed=42)
kernel_flags  = build_kernel_flags(cfg)
spectra_L     = int(cfg.get("synthetic_targets", {}).get("spectra_depth_L", 3))

  Gene features: 5000 genes x 50 dims (expression SVD, 74.8% variance, unit-norm rows)
  DASH kernel factors ON (8/8): weight, ffl, pert_impact, rdf, scber, chi_s, chi_t, rho


## Phase 0: Diagnostic calibration

In [6]:
deg_matrix_csr = None

if FUNGI_MODE == "synthetic":
    from alt_diagnostics import run_synthetic_diagnostics
    utopian_bounds, loss_weights, diagnostic_report, deg_matrix_csr = \
        run_synthetic_diagnostics(
            adata           = adata,
            n_genes         = N_GENES,
            cfg_diagnostics = cfg["diagnostics"],
            cfg_input       = cfg["input"],
            raw_sparse_mat  = raw_sparse_mat,
            spectra_L       = spectra_L,
            gene_features   = gene_features)
else:
    from diagnostics import run_diagnostics
    utopian_bounds, loss_weights, diagnostic_report = run_diagnostics(
        adata           = adata,
        n_genes         = N_GENES,
        cfg_diagnostics = cfg["diagnostics"],
        cfg_input       = cfg["input"],
        raw_sparse_mat  = raw_sparse_mat,
        lambda_user_cfg = cfg.get("lambda_user"),
        bounds_override_cfg = cfg.get("bounds_override"),
        organic_targets_cfg = cfg.get("organic_targets"))

lam_eff = diagnostic_report["lam_eff"]
lam_q25 = diagnostic_report["lam_q25"]
lam_q75 = diagnostic_report["lam_q75"]


Phase 0: Building perturbation impact array...
  Metacell pooling (factor=10): LFC cutoff 0.250 → 0.079
  1,072 perturbation groups detected.


  LFC proxy: 100%|██████████████████████| 1072/1072 [00:00<00:00, 4952.62pert/s]


  Selected 500 perts (100 top-proxy + 400 random tail).


  DEG counts: 100%|█████████████████████████| 500/500 [01:34<00:00,  5.27pert/s]


  Active perts    : 482 / 500 tested
  DEG matrix      : 363,826 causal edges
  LFC matrix      : 500 perturbations × 5000 genes

  λ_target = 40.00 edges/gene  [user-controlled]
  λ_search = [30.00, 50.00]  substrate=30.00  |  erank(GD)=27.14 [diagnostic]

  Running probes...

FUNGI v16.0 — Phase 0 Diagnostic Summary
  Dataset: 5,000 HVGs | 482/500 active perturbations
  λ_target = 40.00 edges/gene  [user-controlled]
  λ_search = [30.00, 50.00]  (probe substrate: 30.00 edges/gene)
  λ_erank  = 27.14  [diagnostic only: Gavish-Donoho effective rank]

  ✓ alpha [2.0920, 2.4182]  conf=0.63  wt=12.6  — Scale-free degree exponent
  ✓  gini [0.5445, 0.8445]  conf=0.70  wt=14.0  — Out-degree inequality (source hub dominance)
  ✓ gini_in [0.5797, 0.7797]  conf=0.68  wt=13.6  — In-degree inequality (regulatory-input concentration)
  ✓     Q [0.0672, 0.1672]  conf=0.70  wt=14.0  — Community modularity (Leiden, resolution=1.3 fixed)
  ✓ S_max [0.1937, 0.2938]  conf=0.70  wt=14.0  — Largest hub's 

In [7]:
# Persist the diagnostic report (underscore keys are large arrays, dropped here)
diag_dir = OUTPUT_ROOT / "phase0_diagnostics"
pd.DataFrame([{k: v for k, v in diagnostic_report.items()
               if not k.startswith("_")}]).to_json(
    diag_dir / "diagnostic_report.json", indent=2)
print(f"Diagnostic report saved to {diag_dir / 'diagnostic_report.json'}")

Diagnostic report saved to ..\DATA\FUNGI_outputs\phase0_diagnostics\diagnostic_report.json


In [8]:
# Synthetic primary: demote source_conc (near-redundant with lambda).
# Windows are left at probe values — SCBER creates the weight heterogeneity
# that satisfies weight_entropy >= 3.8 naturally.
if FUNGI_MODE == "synthetic":
    _demote = list(cfg.get("synthetic_targets", {}).get(
        "demote_to_secondary", ["source_conc"]))
    _dw = float(cfg.get("synthetic_targets", {}).get("demote_weight", 0.0))
    for _k in _demote:
        if _k in loss_weights:
            loss_weights[_k] = _dw
    print("Synthetic primary adjusted:")
    for _k in sorted(utopian_bounds):
        print(f"  {_k:16s} [{utopian_bounds[_k][0]:.4f}, {utopian_bounds[_k][1]:.4f}]"
              f"  w={loss_weights.get(_k, 0.0):.3f}")
else:
    print("Organic mode — no adjustment.")


Organic mode — no adjustment.


In [9]:
# Run the OTHER domain's probes on the dense parent graph to get target-specific
# secondary windows. These override the config fallbacks in the yin-yang secondary.
secondary_windows = {}
if FUNGI_MODE == "synthetic":
    from diagnostics import run_diagnostics as _run_org
    _org_ub, _, _org_rep = _run_org(
        adata, N_GENES, cfg["diagnostics"], cfg["input"],
        raw_sparse_mat=raw_sparse_mat, lambda_user_cfg=cfg.get("lambda_user"))
    # exp_011 Item 3: pull the FINALIZED LIVE biologic windows (gini_out + Q
    # excluded -- disabled in the exp_010 lock) + reciprocity as a fixed prior.
    for _k in ("alpha", "gini_in", "S_max", "C", "rho"):
        if _k in _org_ub:
            secondary_windows[_k] = list(_org_ub[_k])
    secondary_windows["reciprocity"] = [0.02, 0.12]
else:
    from alt_diagnostics import run_synthetic_diagnostics as _run_syn
    _syn_ub, _, _syn_rep, _syn_deg = _run_syn(
        adata, N_GENES, cfg["diagnostics"], cfg["input"],
        raw_sparse_mat=raw_sparse_mat, spectra_L=spectra_L,
        gene_features=gene_features)
    # exp_011 Item 2: DO NOT clobber the FAGCN-ideal heterophily/spectral_gap
    # windows with substrate probe values -- the biology->synthetic pull must
    # aim at the FAGCN ideal (config windows), not at what the substrate already
    # is. Only weight_entropy (a floor) is safe to take from the substrate probe.
    # NOTE (flagged for Patrick): the config FAGCN heterophily window is on the
    # cross-community-FRACTION scale while the loss/probe use feature COSINE-DISTANCE;
    # the window is directionally correct (pull toward higher heterophily) but its
    # absolute bound needs re-derivation on the cosine-distance scale.
    for _k in ("weight_entropy",):
        if _k in _syn_ub:
            secondary_windows[_k] = list(_syn_ub[_k])

print("Probe-derived secondary windows:")
for _k, _v in secondary_windows.items():
    print(f"  {_k:14s} {[round(x, 4) for x in _v]}")


  ═══ Synthetic Phase 0: SPECTRA-targeted diagnostics (L=3) ═══
Phase 0: Building perturbation impact array...
  Metacell pooling (factor=10): LFC cutoff 0.250 → 0.079
  1,072 perturbation groups detected.


  LFC proxy: 100%|██████████████████████| 1072/1072 [00:00<00:00, 4358.74pert/s]


  Selected 500 perts (100 top-proxy + 400 random tail).


  DEG counts: 100%|█████████████████████████| 500/500 [01:38<00:00,  5.10pert/s]


  Active perts    : 482 / 500 tested
  DEG matrix      : 363,826 causal edges
  LFC matrix      : 500 perturbations × 5000 genes

  λ_center = 27.14 edges/gene  [λ_q25=4.83  λ_q75=60.00]
  Active perturbation sources: 397

  Running synthetic probes...
    [1/6] EPR@k          → [0.5342, 1.0000]  (conf=0.79)
    [2/6] Weight entropy → [3.8000, 7.0000]  (conf=0.85)
    [3/6] Path length    → [1.7905, 3.0000]  (conf=0.80)  [hard ceiling=L=3]
    [4/6] Spectral gap   → [0.31691, 0.50000]  (conf=0.72)
    [5/6] Source conc.   → [99.90, 200.00]  (conf=0.82)
    [6/6] Heterophily    → [0.0277, 0.2277]  (conf=0.78)

  ────────────────────────────────────────────────────────────────────────
  SYNTHETIC DIAGNOSTIC SUMMARY  (λ_center=27.14, L=3)
  ────────────────────────────────────────────────────────────────────────
  Parameter             Target window         Conf    Weight   Description
  ────────────────────────────────────────────────────────────────────────
  epr_k                 [0.53

## Phase 2: Prefilter and edge weights

In [10]:
from scipy.stats import rankdata
from filtering import adaptive_threshold_filter

G_work = adaptive_threshold_filter(
    raw_sparse_mat, target_density=cfg["prefilter"]["target_density"])
gc.collect()

G_work_coo  = G_work.tocoo()
sources_raw = G_work_coo.row.copy()
targets_raw = G_work_coo.col.copy()
weights_raw = G_work_coo.data.copy()
print(f"Working graph: {G_work.nnz:,} edges  (density {G_work.nnz/N_GENES**2:.4%})")

Filtering graph to 25.0% density...
  Filtered to 6,250,000 edges (25.0000% density).
Working graph: 6,250,000 edges  (density 25.0000%)


In [11]:
# Rank normalise weights for the kernel; log scale raw weights for the output
W_ranked = rankdata(weights_raw, method="average").astype(np.float64) / len(weights_raw)
if W_ranked.max() - W_ranked.min() < 1e-6:
    print("WARNING: degenerate weights, using log1p fallback")
    W_log = np.log1p(weights_raw.astype(np.float64))
    W_arr = (np.clip(W_log / max(W_log.max(), 1e-10), 0.001, 1.0)
             if W_log.max() > 1e-12
             else np.random.default_rng(42).uniform(0.01, 1.0, len(weights_raw)))
else:
    W_arr = W_ranked

_w_raw_max = float(weights_raw.max())
W_spectra  = (np.log1p(weights_raw.astype(np.float64)) / np.log1p(_w_raw_max)
              if _w_raw_max > 0 else W_arr.copy())
_spectra_weight_lookup = {
    (int(s), int(t)): float(w)
    for s, t, w in zip(sources_raw, targets_raw, W_spectra)}

sources_arr = sources_raw.astype(np.int32)
targets_arr = targets_raw.astype(np.int32)
print(f"W_arr range    : [{W_arr.min():.4f}, {W_arr.max():.4f}]")
print(f"W_spectra range: [{W_spectra.min():.4f}, {W_spectra.max():.4f}]")

W_arr range    : [0.0018, 1.0000]
W_spectra range: [0.0476, 1.0000]


In [12]:
# Per source quantile weights, hub caps, and the source perturbation impact
from engine import (compute_source_quantile_weights,
                    compute_pagerank_kappa_multipliers,
                    compute_source_pert_impact)

W_source_quantile = compute_source_quantile_weights(
    sources_arr, W_arr, N_GENES)
D_arr = np.zeros(len(W_arr), dtype=np.float64)

_pkc = cfg["pagerank_kappa"]
per_gene_kappa = compute_pagerank_kappa_multipliers(
    G_work, N_GENES,
    alpha=_pkc.get("alpha", 0.85),
    n_iter=_pkc.get("n_iter", 60),
    hub_percentile=_pkc.get("hub_percentile", 99.0),
    hub_multiplier=_pkc.get("hub_multiplier", 3.0))

# The diagnostic report stores the impact array as a list for JSON; wrap it in a
# float array before it is used with boolean indexing inside the engine.
source_pert_impact = compute_source_pert_impact(
    np.array(diagnostic_report["_impact_array"], dtype=np.float64),
    diagnostic_report["_perturbation_labels"],
    diagnostic_report["_name_to_idx"],
    N_GENES)
print(f"PageRank kappa : range [{per_gene_kappa.min():.3f}, {per_gene_kappa.max():.3f}]")

PageRank kappa : range [1.000, 3.000]


## Phase 2: Biological priors and SCBER

In [13]:
from engine import compute_chi_prior, compute_deg_row_prior

chi_cfg     = cfg.get("chi_prior", {})
chi_enabled = chi_cfg.get("enabled", True)
zeta        = float(chi_cfg.get("zeta", 1.0))

deg_col_sums   = np.array(diagnostic_report.get("_deg_col_sums", []), dtype=np.float64)
deg_row_sums   = np.array(diagnostic_report.get("_deg_row_sums", []), dtype=np.float64)
deg_out_parent = np.asarray(raw_sparse_mat.sum(axis=1)).ravel().astype(np.float64)

if chi_enabled and len(deg_col_sums) == N_GENES:
    chi_prior = compute_chi_prior(deg_col_sums, N_GENES, zeta=zeta)
    print(f"chi prior : {int((chi_prior>1.05).sum()):,} genes boosted  "
          f"range [{chi_prior.min():.3f}, {chi_prior.max():.3f}]")
else:
    chi_prior = None
    print("chi prior : disabled")

if len(deg_row_sums) == N_GENES:
    rho_prior = compute_deg_row_prior(
        deg_row_sums        = deg_row_sums,
        deg_out_parent      = deg_out_parent,
        perturbation_labels = np.array(diagnostic_report["_perturbation_labels"]),
        name_to_idx         = diagnostic_report["_name_to_idx"],
        n_genes             = N_GENES)
    print(f"rho prior : range [{rho_prior.min():.3f}, {rho_prior.max():.3f}]")
else:
    rho_prior = None
    print("rho prior : skipped")
chi_t_prior = None

chi prior : 1,683 genes boosted  range [1.000, 1.398]
  rho_prior: 153 genes boosted above 1.05 (range [1.000, 2.113])
rho prior : range [1.000, 2.113]


In [14]:
from effective_resistance import compute_scber_scores

er_cfg = cfg.get("effective_resistance", {})
if er_cfg.get("enabled", True):
    er_normalized, inter_mask, er_raw, scber_diag = compute_scber_scores(
        G_work, sources_arr, targets_arr,
        cfg=er_cfg,
        gene_features=gene_features)
    er_eta = float(er_cfg.get("eta_inter", 0.20))
    # session 5: m_intra's static per-edge factor and Q's loss-time community
    # lookup both reuse SCBER's own Leiden membership -- never a second
    # community-detection run.
    gene_community_labels = scber_diag.get("membership") if scber_diag else None
    intra_mask = (~inter_mask) if inter_mask is not None else None
else:
    er_normalized, inter_mask, er_eta, scber_diag = None, None, 0.20, None
    gene_community_labels, intra_mask = None, None
    print("SCBER : disabled")


    Running Leiden community detection on G_work...
    Leiden: resolution=1.3, Q=0.026, communities=308, largest_community_fraction=0.170
    Inter-module edges: 5,473,273 (87.6%)  Intra-module: 776,727 (12.4%)
    Computing ER sketch: k=300, ε=0.35
    SCBER at η_inter=0.28:
      Inter-module factor: [0.432, 1.000] (mean 0.811)
      Intra-module factor: 1.000 exactly (untouched)
      High-ER bridges (R>0.9): 652,143 edges


## Regulatory Diversity Factor (RDF) prior

In [15]:
from engine import compute_rdf_prior

rdf_prior = compute_rdf_prior(
    sources_arr, targets_arr, W_arr, gene_features, N_GENES, top_n=50)


  rdf_prior: 1380 genes diverse (RDF>0.7), 3620 bloc-like (RDF<0.3), range [0.100, 1.000]


## Phase 3: Expansive search

In [16]:
from search import generate_sobol_samples, SearchEvaluator
from diagnostics import build_shatter_config

es_cfg = cfg["expansive_search"]
hp_cfg = cfg["hyperparameter_bounds"]

sobol_params, lower_bounds, upper_bounds = generate_sobol_samples(
    n_genes=N_GENES, n_samples=es_cfg["n_samples"], hp_cfg=hp_cfg,
    seed=es_cfg["random_seed"], lam_eff=lam_eff,
    lam_q25=lam_q25, lam_q75=lam_q75)

pert_col   = cfg["input"]["perturbation_column"]
ctrl_label = cfg["input"]["control_label"]
pert_genes = [g for g in adata.obs[pert_col].unique() if g != ctrl_label]
gene_list  = list(adata.var_names)
perturbed_nodes = np.array(
    [gene_list.index(g) for g in pert_genes
     if g in gene_list and gene_list.index(g) < N_GENES], dtype=int)
print(f"Perturbation targets in gene universe: {len(perturbed_nodes):,}")

  Sobol sequence: 8,192 points in 8D space.
  Parameter space: β[0.5,4.0]  δ[0.00,3.00]  κ[0.020,0.350]  k_core[8,28]  λ[30.0,50.0]  ψ[0.0,3.0]  ν[0.0,2.0]  m_intra[1.00,2.50]
Perturbation targets in gene universe: 868


In [17]:
shatter_cfg = build_shatter_config(
    cfg["shatter"], N_GENES, utopian_bounds,
    [lower_bounds[4], upper_bounds[4]], mode=FUNGI_MODE)
# synthetic path length gate threshold (ignored in organic mode)
shatter_cfg["min_reach_frac"] = cfg.get("synthetic_targets", {}).get("min_reach_frac", 0.5)

In [18]:
gpu_cfg = cfg.get("gpu", {})
evaluator = SearchEvaluator(
    W_arr           = W_arr,
    W_q_arr         = W_source_quantile,
    D_arr           = D_arr,
    sources_arr     = sources_arr,
    targets_arr     = targets_arr,
    n_genes         = N_GENES,
    perturbed_nodes = perturbed_nodes,
    utopian_bounds  = utopian_bounds,
    loss_weights    = loss_weights,
    shatter_cfg     = shatter_cfg,
    per_gene_kappa  = per_gene_kappa,
    source_pert_impact = source_pert_impact,
    n_workers       = es_cfg["n_workers"],
    src_dir         = SRC_DIR,
    er_scores       = er_normalized,
    er_eta          = er_eta,
    inter_mask      = inter_mask,
    intra_mask      = intra_mask,
    chi_prior       = chi_prior,
    rho_prior       = rho_prior,
    chi_t_prior     = chi_t_prior,
    rdf_prior       = rdf_prior,
    mode            = FUNGI_MODE,
    deg_matrix_csr  = deg_matrix_csr,
    gene_community_labels = gene_community_labels,
    kernel_flags    = kernel_flags,
    gene_features   = gene_features,
    spectra_L       = spectra_L,
    exact_spectral  = False,
    use_ray_pipeline = gpu_cfg.get("use_ray_pipeline", True))


In [19]:
# IMPORTANT: init_gpu_context's docstring requires W/W_q/sources/targets/
# er_scores/inter_mask to already be "globally presorted by descending
# weight" -- it does NOT sort internally, it just truncates to Ne. Must be
# built from evaluator's OWN presorted arrays (evaluator.Ws/Wqs/srcs/tgts/
# er_scores/inter_mask), not the raw W_arr/sources_arr/targets_arr/
# er_normalized/inter_mask -- otherwise the GPU context's Ne-truncated
# candidate pool silently becomes a different (effectively arbitrary,
# row-major-ordered, NOT actually top-weight) ~550k-edge subset than the
# one build_graph_from_params uses for cohort/champion export. Confirmed
# empirically: the two pools shared only ~22% overlap when this bug was
# present, meaning every GPU-scored eval during Phase 3/5 was being judged
# against the wrong edge pool, not just the final champion export. See
# markdowns/claude_code/claude_code_session_3.md for the full investigation.
# session 5: intra_mask (m_intra's static per-edge factor) follows the exact
# same presort discipline as inter_mask -- evaluator.intra_mask, not the raw
# intra_mask computed in the SCBER cell.
if gpu_cfg.get("enabled", True):
    from engine import init_gpu_context, release_gpu_context
    release_gpu_context()  # clear any context from a previous cell re-run
    gpu_ctx = init_gpu_context(
        W=evaluator.Ws, W_q=evaluator.Wqs,
        sources=evaluator.srcs, targets=evaluator.tgts,
        n_genes=N_GENES, shatter_cfg=shatter_cfg,
        source_pert_impact=source_pert_impact, k_core_bounds=hp_cfg["k_core"],
        rdf_prior=rdf_prior,
        er_scores=evaluator.er_scores, er_eta=er_eta, inter_mask=evaluator.inter_mask,
        intra_mask=evaluator.intra_mask,
        chi_prior=chi_prior, chi_t_prior=chi_t_prior, rho_prior=rho_prior,
        kernel_flags=kernel_flags, n_ffl_buckets=gpu_cfg.get("n_ffl_buckets", 8))
    if not gpu_ctx.enabled:
        print("[FUNGI GPU] CUDA unavailable -- evaluator.evaluate() will use CPU/Ray.")
else:
    from engine import release_gpu_context
    release_gpu_context()
    print("GPU acceleration disabled in config -- using CPU/Ray.")


[FUNGI GPU] Context initialized: Ne=550,000 n_genes=5,000 max_seg_len=3,785 VRAM=0.05 GB


In [20]:
shard_dir = str(OUTPUT_ROOT / "phase3_expansive_search" / "shards")
df_expansive = evaluator.evaluate(
    list(sobol_params), chunk_size=es_cfg["chunk_size"],
    shard_dir=shard_dir, desc=f"Phase 3 [{FUNGI_MODE}]")

n_viable = int((df_expansive["is_shattered"] == 0).sum())
best = df_expansive.loc[df_expansive["is_shattered"] == 0, "utopia_loss"].min() \
    if n_viable else float("inf")
print(f"\nExpansive search: {len(df_expansive):,} graphs")
print(f"  Viable    : {n_viable:,}")
print(f"  Shattered : {len(df_expansive)-n_viable:,} "
      f"({(1-n_viable/max(len(df_expansive),1))*100:.1f}%)")
print(f"  Best loss : {best:.4f}")

if n_viable == 0:
    raise RuntimeError("No viable graphs from Phase 3, cannot proceed.")

c:\Users\studi\OneDrive\Documents\thesis\fungi_bot\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Phase 3 [organic]:   0%|          | 0/8192 [00:00<?, ?graph/s]


Expansive search: 8,192 graphs
  Viable    : 8,192
  Shattered : 0 (0.0%)
  Best loss : 0.1809


## Phase 5: Refinement

In [21]:
from refinement import run_ml_gmm_refinement

if deg_matrix_csr is None:
    from diagnostics import build_impact_array
    print("Building DEG matrix for precision scoring...")
    (_, _, _, deg_matrix_csr, _, _, _, _) = build_impact_array(
        adata,
        perturbation_column     = cfg["input"]["perturbation_column"],
        control_label           = cfg["input"]["control_label"],
        is_metacell             = cfg["input"]["is_metacell"],
        metacell_pooling_factor = cfg["input"]["metacell_pooling_factor"])
    print(f"  DEG matrix: {deg_matrix_csr.nnz:,} causal edges  {deg_matrix_csr.shape}")
else:
    print(f"Reusing Phase 0 DEG matrix: {deg_matrix_csr.nnz:,} causal edges")

df_refinement, was_skipped = run_ml_gmm_refinement(
    df_phase3         = df_expansive,
    lower             = lower_bounds,
    upper             = upper_bounds,
    evaluator         = evaluator,
    refinement_cfg    = cfg["refinement"],
    verbose           = True,
    deg_matrix        = deg_matrix_csr,
    mode              = FUNGI_MODE,
    gene_features     = gene_features,
    kernel_flags      = kernel_flags,
    spectra_L         = spectra_L,
    secondary_cfg     = cfg.get("secondary_objective"),
    secondary_windows = secondary_windows)

if df_refinement is not None:
    df_refinement.to_csv(
        OUTPUT_ROOT / "phase5_refinement" / "refinement_results.csv", index=False)
    n_ref = int((df_refinement["is_shattered"] == 0).sum())
    print(f"Refinement: {len(df_refinement):,} evaluations | {n_ref:,} viable")
else:
    print("Refinement skipped")


Building DEG matrix for precision scoring...
Phase 0: Building perturbation impact array...
  Metacell pooling (factor=10): LFC cutoff 0.250 → 0.079
  1,072 perturbation groups detected.


  LFC proxy:   0%|                                   | 0/1072 [00:00<?, ?pert/s]

  LFC proxy: 100%|██████████████████████| 1072/1072 [00:00<00:00, 3481.52pert/s]


  Selected 500 perts (100 top-proxy + 400 random tail).


  DEG counts: 100%|█████████████████████████| 500/500 [01:48<00:00,  4.59pert/s]


...

       Region 4 [loss]: no gain (score=0.8600, loss=0.1628, radius=0.375, viable=256/256)


  Region 5 r3 [loss]:   0%|          | 0/256 [00:00<?, ?graph/s]

       Region 5 [loss]: no gain (score=0.8572, loss=0.1666, radius=0.375, viable=256/256)


  Region 6 r3 [loss]:   0%|          | 0/256 [00:00<?, ?graph/s]

       Region 6 [loss]: no gain (score=0.8586, loss=0.1647, radius=0.125, viable=256/256)
       [Re-detection] no better region found.

  ── Refinement complete: 5,004 evals in 1385.1s
     Best loss: 0.160174 | zero-loss: 0 | regions: 6
       Region 1 [loss]: score=0.8550 (loss=0.1695), best=249,716 edges, rounds=3
       Region 2 [loss]: score=0.8584 (loss=0.1649), best=248,813 edges, rounds=3
       Region 3 [loss]: score=0.8555 (loss=0.1688), best=249,954 edges, rounds=3
       Region 4 [loss]: score=0.8600 (loss=0.1628), best=217,062 edges, rounds=3
       Region 5 [loss]: score=0.8572 (loss=0.1666), best=248,785 edges, rounds=3
       Region 6 [loss]: score=0.8586 (loss=0.1647), best=249,754 edges, rounds=3
Refinement: 5,004 evaluations | 5,004 viable


## Phase 6: Champion selection

In [22]:
from refinement import select_diverse_cohort, compute_epr_at_k
from engine import build_graph_from_params

frames = [df_expansive]
if df_refinement is not None:
    frames.append(df_refinement)
df_all = pd.concat(frames, ignore_index=True)

cohort = select_diverse_cohort(
    df_all, utopian_bounds, N_GENES,
    cohort_size       = 5,
    mode              = FUNGI_MODE,
    evaluator         = evaluator,
    gene_features     = gene_features,
    kernel_flags      = kernel_flags,
    spectra_L         = spectra_L,
    deg_matrix        = deg_matrix_csr,
    secondary_cfg     = cfg.get("secondary_objective"),
    secondary_windows = secondary_windows)

# evaluator.Ws/Wqs/Ds/srcs/tgts are already presorted by
# SearchEvaluator.__init__ -- use directly. Re-presorting here used to be
# able to land on a different Ne-truncated candidate pool than
# init_gpu_context's (see markdowns/claude_code/claude_code_session_3.md);
# presort_edges is now idempotent, but there's no reason to pay the
# O(N log N) cost again either.
_Ws, _Wqs, _Ds, _srcs, _tgts = (
    evaluator.Ws, evaluator.Wqs, evaluator.Ds, evaluator.srcs, evaluator.tgts)


In [23]:
# Rebuild each cohort graph with the configured kernel and score its precision
_cohort_epr  = {}
_cohort_bufs = {}
for _, row in cohort.iterrows():
    rank   = int(row["cohort_rank"])
    params = tuple(row[["beta","delta","kappa","k_core","lambda","psi","nu","m_intra"]].values.astype(float))
    try:
        ss, st, sw = build_graph_from_params(
            params, _Ws, _Wqs, _Ds, _srcs, _tgts,
            N_GENES, perturbed_nodes, shatter_cfg,
            per_gene_kappa, source_pert_impact,
            md_gate    = evaluator.md_gate,
            er_scores  = evaluator.er_scores,
            er_eta     = evaluator.er_eta,
            inter_mask = evaluator.inter_mask,
            intra_mask = evaluator.intra_mask,
            chi_prior  = evaluator.chi_prior,
            rho_prior  = evaluator.rho_prior,
            chi_t_prior = evaluator.chi_t_prior,
            rdf_prior   = evaluator.rdf_prior,
            kernel_flags = kernel_flags)
        _cohort_bufs[rank] = (ss, st, sw)
        _cohort_epr[rank]  = compute_epr_at_k(ss, st, deg_matrix_csr, perturbed_nodes)
    except Exception as e:
        print(f"  cohort rank {rank}: rebuild failed ({e})")
        _cohort_bufs[rank] = (None, None, None)
        _cohort_epr[rank]  = 0.0
print(f"Rebuilt {len(_cohort_bufs)} cohort graphs")


Rebuilt 5 cohort graphs


In [24]:
# Champion audit � compute metrics from the rebuilt graph, not the result row.
# run_dash_and_score does NOT emit synthetic columns into the result DataFrame.
import math
from refinement import _secondary_metrics

champ     = cohort[cohort["is_champion"]].iloc[0]
champ_ss, champ_st, champ_sw = _cohort_bufs.get(1, (None, None, None))

print("Champion topology vs targets:")
if champ_ss is not None:
    _champ_m = _secondary_metrics(
        champ_ss, champ_st, champ_sw,
        N_GENES, perturbed_nodes, deg_matrix_csr, gene_features,
        want={"epr_k","weight_entropy","spectral_gap","source_conc",
              "heterophily","gini","gini_in","rho"},
        need_spectral=True)
    _champ_m["n_edges"] = len(champ_ss)

    if FUNGI_MODE != "synthetic":
        _audit = ["alpha","gini","gini_in","Q","S_max","C","rho"]
    else:
        _audit = ["epr_k","weight_entropy","spectral_gap","source_conc","heterophily"]

    n_pass = 0
    for _p in _audit:
        if _p not in utopian_bounds:
            continue
        lo, hi = utopian_bounds[_p]
        # try rebuilt metrics first, fall back to result row for organic cols
        val = _champ_m.get(_p, float(champ.get({"gini":"Gini"}.get(_p, _p), float("nan"))))
        ok  = (lo <= val <= hi) if not math.isnan(val) else False
        n_pass += int(ok)
        print(f"  {'PASS' if ok else 'miss'} {_p:>14s} = {val:.4f}  [{lo:.4f}, {hi:.4f}]")
    print(f"  {n_pass}/{len(_audit)} targets satisfied")
    print(f"  n_edges: {len(champ_ss):,}  |  EPR@k: {_champ_m.get('epr_k',0):.4f}")
else:
    print("  Champion rebuild failed.")


Champion topology vs targets:
  PASS          alpha = 2.3022  [2.0920, 2.4182]
  miss           gini = 0.8921  [0.5445, 0.8445]
  PASS        gini_in = 0.5957  [0.5797, 0.7797]
  PASS              Q = 0.1579  [0.0672, 0.1672]
  miss          S_max = 0.1924  [0.1937, 0.2938]
  PASS              C = 0.1163  [0.1066, 0.1666]
  PASS            rho = -0.1216  [-0.3117, 0.0000]
  5/7 targets satisfied
  n_edges: 215,225  |  EPR@k: 0.6379


In [25]:
# Secondary metrics for every cohort rank (not just the champion) -- needed by
# report.py's Section 8 (Cohort Overview) and Section 16 (Alternate Graph
# Comparison) tables. _secondary_metrics is already imported above.
cohort_secondary_metrics = {}
for _rank, (_ss, _st, _sw) in _cohort_bufs.items():
    if _ss is None:
        cohort_secondary_metrics[_rank] = {}
        continue
    try:
        _m = _secondary_metrics(
            _ss, _st, _sw,
            N_GENES, perturbed_nodes, deg_matrix_csr, gene_features,
            want={"epr_k", "weight_entropy", "spectral_gap", "source_conc",
                  "heterophily", "gini", "gini_in", "rho"},
            need_spectral=True)
        _m["n_edges"] = len(_ss)
        cohort_secondary_metrics[_rank] = _m
    except Exception as e:
        print(f"  cohort rank {_rank}: secondary metrics failed ({e})")
        cohort_secondary_metrics[_rank] = {}
print(f"Computed secondary metrics for {len(cohort_secondary_metrics)} cohort graphs")

Computed secondary metrics for 5 cohort graphs


## Phase 6: Cohort summary table

In [26]:
# Cohort summary table: all 5 graphs side-by-side -- hyperparameters (inputs)
# + mode-aware output parameters + EPR@k + edge count. Restores the readout
# that used to exist before cohort export was changed to save all 5 graphs
# by default.
#
# Output parameters come from the STORED search-time result row (cohort's
# own alpha/Gini/gini_in/Q/S_max/C/rho columns), NOT a fresh rebuild -- investigation
# this session found build_graph_from_params does not always reproduce the
# exact graph that was scored during search/refinement for the same
# hyperparameters (see markdowns/claude_code/claude_code_session_3.md), so
# preferring the rebuild here (as the old single-champion audit cell did)
# can silently show the wrong topology. EPR@k is the one metric that
# genuinely requires the rebuilt edges -- it's never part of the stored loss.
_PARAM_COLS = ["beta", "delta", "kappa", "k_core", "lambda", "psi", "nu", "m_intra"]
_rows = []
for _, _row in cohort.sort_values("cohort_rank").iterrows():
    _rank = int(_row["cohort_rank"])
    _entry = {"rank": _rank, "champion": bool(_row.get("is_champion", _rank == 1)),
              "utopia_loss": float(_row["utopia_loss"])}
    for _hp in _PARAM_COLS:
        _entry[_hp] = float(_row[_hp])
    if FUNGI_MODE != "synthetic":
        for _p, _col in [("alpha", "alpha"), ("gini", "Gini"), ("gini_in", "gini_in"),
                          ("Q", "Q"), ("S_max", "S_max"), ("C", "C"), ("rho", "rho")]:
            _entry[_p] = float(_row.get(_col, float("nan")))
    else:
        _sm = cohort_secondary_metrics.get(_rank, {})
        for _p in ["epr_k", "weight_entropy", "spectral_gap", "source_conc", "heterophily"]:
            _entry[_p] = _sm.get(_p, float("nan"))
    _entry["EPR@k"] = cohort_secondary_metrics.get(_rank, {}).get(
        "epr_k", _cohort_epr.get(_rank, float("nan")))
    _entry["n_edges"] = int(_row["n_edges"])
    _rows.append(_entry)

cohort_summary = pd.DataFrame(_rows)
print(f"Cohort summary ({FUNGI_MODE.upper()} mode):")
display(cohort_summary.round(4))


Cohort summary (ORGANIC mode):


,rank,champion,utopia_loss,beta,delta,kappa,k_core,lambda,psi,nu,m_intra,alpha,gini,gini_in,Q,S_max,C,rho,EPR@k,n_edges
0,1,True,0.1602,2.9340,0.0000,0.3500,23.4062,43.0450,0.0000,0.0000,1.8627,2.3022,0.8921,0.5957,0.1579,0.1924,0.1164,-0.1216,0.6379,215225
1,2,False,0.3294,4.0000,3.0000,0.2795,12.0000,30.0000,0.6696,0.4890,1.0773,2.3145,0.9307,0.6373,0.1652,0.2888,0.1124,-0.2563,0.7340,150000
2,3,False,0.2954,0.9520,3.0000,0.3500,23.7262,38.8006,0.0000,0.0000,2.5000,2.2214,0.9141,0.5725,0.1768,0.2716,0.1056,-0.1811,0.6703,194003
3,4,False,0.1945,3.7350,3.0000,0.3216,25.3181,47.8180,0.3601,0.0541,1.0746,2.3389,0.9008,0.6165,0.1290,0.2348,0.1374,-0.1434,0.6397,239090
4,5,False,0.3215,3.9611,2.4717,0.2433,15.9287,39.8217,0.6122,0.0000,2.5000,2.3610,0.9122,0.6009,0.1649,0.2708,0.1280,-0.1905,0.6667,199109


In [27]:
# Synthetic only: recompute the exact spectral gap on the champion for the record
if FUNGI_MODE == "synthetic" and champ_ss is not None \
        and cfg.get("synthetic_targets", {}).get("exact_spectral_finalists", True):
    import scipy.sparse as sp
    from scipy.sparse.linalg import eigsh
    nec = len(champ_ss)
    A = sp.csr_matrix((np.ones(nec), (champ_ss, champ_st)), shape=(N_GENES, N_GENES))
    A = (A + A.T).multiply(0.5); A.eliminate_zeros()
    dv = np.array(A.sum(axis=1)).ravel(); dv[dv == 0] = 1.0
    L = sp.eye(N_GENES, format="csr") - sp.diags(1/np.sqrt(dv)) @ A @ sp.diags(1/np.sqrt(dv))
    eigs = np.sort(np.abs(eigsh(L, k=4, which="SM",
                                return_eigenvectors=False, tol=1e-3, maxiter=500)))
    print(f"Champion exact spectral gap (lambda_2): {eigs[1]:.5f}")

## Phase 6: Graph output

In [28]:
gene_names = list(adata.var_names)
output_dir = OUTPUT_ROOT / "phase6_champion"
mode_tag   = "_synthetic" if FUNGI_MODE == "synthetic" else ""
graph_stem = Path(RAW_GRAPH_PATH.stem).stem

graphs_to_output = [int(r) for r in cohort["cohort_rank"].tolist()]
graph_df = None  # captured below for the champion -- consumed by Phase 7 report

for rank_num in graphs_to_output:
    row = cohort[cohort["cohort_rank"] == rank_num]
    if row.empty:
        continue
    row = row.iloc[0]
    surv_s, surv_t, surv_W = _cohort_bufs.get(rank_num, (None, None, None))
    if surv_s is None:
        continue

    spectra_weights = np.array(
        [_spectra_weight_lookup.get((int(s), int(t)), 0.0)
         for s, t in zip(surv_s.tolist(), surv_t.tolist())], dtype=np.float32)

    is_champ = bool(row.get("is_champion", rank_num == 1))
    suffix   = "" if is_champ else f"_alt{rank_num}"
    tag      = f"{graph_stem}{mode_tag}{suffix}"

    out_df = pd.DataFrame({
        "Regulator": [gene_names[i] for i in surv_s],
        "Target":    [gene_names[i] for i in surv_t],
        "Weight":    spectra_weights})
    out_path = output_dir / f"{tag}.parquet"
    out_df.to_parquet(out_path, index=False)
    print(f"  saved {tag}: {len(out_df):,} edges -> {out_path.name}")

    if is_champ:
        graph_df = out_df

  saved RPE1_5k_essential_selftrain_dense_graph: 215,225 edges -> RPE1_5k_essential_selftrain_dense_graph.parquet
  saved RPE1_5k_essential_selftrain_dense_graph_alt2: 150,000 edges -> RPE1_5k_essential_selftrain_dense_graph_alt2.parquet
  saved RPE1_5k_essential_selftrain_dense_graph_alt3: 194,003 edges -> RPE1_5k_essential_selftrain_dense_graph_alt3.parquet
  saved RPE1_5k_essential_selftrain_dense_graph_alt4: 239,090 edges -> RPE1_5k_essential_selftrain_dense_graph_alt4.parquet
  saved RPE1_5k_essential_selftrain_dense_graph_alt5: 199,109 edges -> RPE1_5k_essential_selftrain_dense_graph_alt5.parquet


In [29]:
# Sanity check the output weight column before handing to the GNN
import glob
for p in sorted(glob.glob(str(output_dir / f"{graph_stem}{mode_tag}*.parquet"))):
    d = pd.read_parquet(p)
    q25 = d["Weight"].quantile(0.25)
    flag = "OK" if 0.1 < q25 < 0.7 else "CHECK (expected log scaled gains)"
    print(f"  {Path(p).name}: p25 weight = {q25:.3f}  {flag}")

  RPE1_5k_essential_selftrain_dense_graph.parquet: p25 weight = 0.497  OK
  RPE1_5k_essential_selftrain_dense_graph_alt2.parquet: p25 weight = 0.540  OK
  RPE1_5k_essential_selftrain_dense_graph_alt3.parquet: p25 weight = 0.496  OK
  RPE1_5k_essential_selftrain_dense_graph_alt4.parquet: p25 weight = 0.515  OK
  RPE1_5k_essential_selftrain_dense_graph_alt5.parquet: p25 weight = 0.514  OK


## Phase 7: Report generation

In [30]:
from report import generate_report

reports_dir = OUTPUT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
run_tag = f"{graph_stem}{mode_tag}"

# graphs_to_report=[1] (champion only): report.py's per-graph deep-dive
# sections (9F-15) all read from the single graph_df passed in here, so
# reporting on alternates too would repeat the champion's own graph/topology
# under their headings -- report.py has no per-rank graph_df slot.
generate_report(
    run_name                 = run_tag,
    cfg                      = cfg,
    adata                    = adata,
    graph_df                 = graph_df,
    cohort                   = cohort,
    utopian_bounds           = utopian_bounds,
    loss_weights             = loss_weights,
    diagnostic_report        = diagnostic_report,
    experimental_df          = experimental_df,
    df_expansive             = df_expansive,
    df_refinement            = df_refinement,
    sources_arr              = evaluator.srcs,
    targets_arr              = evaluator.tgts,
    W_arr                    = W_arr,
    source_pert_impact       = source_pert_impact,
    chi_prior                = evaluator.chi_prior,
    rho_prior                = evaluator.rho_prior,
    total_gate               = None,
    er_diagnostics           = scber_diag,
    pert_efficiency_map      = None,
    alpha_md                 = 0.5,
    alpha_stab               = 0.3,
    output_dir               = reports_dir,
    graphs_to_report         = [1],
    fungi_mode               = FUNGI_MODE,
    fingerprint              = None,
    champ_secondary_metrics  = cohort_secondary_metrics.get(1, {}),
    cohort_secondary_metrics = cohort_secondary_metrics,
)
print(f"Report written to {reports_dir}")

FUNGI report saved → ..\DATA\FUNGI_outputs\reports\RPE1_5k_essential_selftrain_dense_graph_report.md
Report written to ..\DATA\FUNGI_outputs\reports


## Optional cleanup

In [31]:
import shutil
import time

CLEANUP = True
if CLEANUP:
    for phase_name in ["phase1_ingestion","phase2_normalization",
                       "phase3_expansive_search","phase5_refinement"]:
        d = OUTPUT_ROOT / phase_name
        if not d.exists():
            continue
        # OneDrive sync (this whole project lives under OneDrive) or AV/indexer
        # scanning can transiently hold a file open right after a phase finishes
        # writing -- shutil.rmtree has no retry, so one locked file crashes the
        # whole cell and leaves cleanup half-done. Retry a few times, then warn
        # and move on to the next phase rather than aborting.
        for attempt in range(4):
            try:
                shutil.rmtree(d)
                print(f"  deleted {phase_name}/")
                break
            except PermissionError as e:
                if attempt < 3:
                    print(f"  {phase_name}/ locked, retrying in 3s... ({e.filename})")
                    time.sleep(3)
                else:
                    print(f"  WARNING: could not delete {phase_name}/ -- still "
                          f"locked ({e.filename}). Skipping -- delete manually "
                          f"later (e.g. once OneDrive sync settles).")
    print("Cleanup complete. Preserved phase6_champion/ and phase0_diagnostics/")
else:
    print("Cleanup skipped. Set CLEANUP=True to free disk space.")

  deleted phase1_ingestion/
  deleted phase2_normalization/
  deleted phase3_expansive_search/
  deleted phase5_refinement/
Cleanup complete. Preserved phase6_champion/ and phase0_diagnostics/
